In [1]:
import numpy as np

In [68]:
# Activation functions

def relu(x):
    return np.maximum(0,x)

def sigmoid(x):
    return 1/(1+np.exp(-x))

# 2 features (X) with 5 samples (m)
# NN with 3 hidden layers (4, 3, 1(output layer))

In [76]:
np.random.seed(123)

X = np.random.randn(4, 5) # (features, samples)
y = (np.random.randn(1, 5) > 0)

hidden_nodes = []
h1_nodes = 4 # number of nodes in hidden layer 1
h2_nodes = 3 # number of nodes in hidden layer 2
hidden_nodes.append(h1_nodes) 
hidden_nodes.append(h2_nodes) #hidden_nodes = [4, 3]

In [77]:
def layer_sizes(X, y, hidden_nodes):
    n_x0 = X.shape[0] # number of features in X
    n_x1 = hidden_nodes[0]
    n_x2 = hidden_nodes[1]
    n_y = y.shape[0]

    return n_x0, n_x1, n_x2, n_y

In [65]:
def int_params(n_x0, n_x1, n_x2, n_y):
    w1 = (np.random.randn(n_x1, n_x0)) 
    w2 = (np.random.randn(n_x2, n_x1)) 
    w3 = (np.random.randn(n_y, n_x2)) 
    # trick for w (number of nodes in current layer, number of nodes in prev layer)
    # for first w just take (number of nodes in current layer, number of features)

    b1 = np.random.randn(n_x1, 1)
    b2 = np.random.randn(n_x2, 1)
    b3 = np.random.randn(n_y, 1)
    
    # trick for w (number of nodes in current layer, 1)

    params = {"w1": w1, "b1": b1, "w2": w2, "b2": b2,  "w3": w3, "b3": b3}
    return params

In [74]:
def fwp(X, params):
    w1 = params["w1"]
    b1 = params["b1"]
    w2 = params["w2"]
    b2 = params["b2"]
    w3 = params["w3"]
    b3 = params["b3"]

    # layer 1
    z1 = w1 @ X + b1 # same as np.dot(w1, X) + b
    a1 = np.tanh(z1)

    #layer 2
    z2 = w2 @ a1 + b2
    a2 = relu(z2)

    #layer 3
    z3 = w3 @ a2 + b3
    a3 = sigmoid(z3)
    # a3 = np.clip(a3, 1e-15, 1 - 1e-15) #optional

    cache = { "z1": z1, "a1": a1, "z2": z2, "a2": a2,"z3": z3, "a3": a3}
    return a3, cache

In [19]:
def compute_cost(y, a3):
    m = y.shape[1] # total number of samples
    cost1 = np.sum(y * np.log(a3) + (1 - y) * np.log(1 - a3))
    cost = -cost1 / m
    cost = float(np.squeeze(cost))
    return cost

In [61]:
def bwp(X, y, params, cache):
    w1 = params["w1"]
    b1 = params["b1"]
    w2 = params["w2"]
    b2 = params["b2"]
    w3 = params["w3"]
    b3 = params["b3"]

    a1 = cache["a1"]
    a2 = cache["a2"]
    a3 = cache["a3"]

    z2 = cache["z2"]

    m = y.size

    # last layer (sigmoid)
    dz3 = a3 - y
    dw3 = (dz3 @ a2.T) / m
    db3 = np.sum(dz3, axis=1, keepdims=True)

    # 2nd layer (relu)
    da2 = w3.T @ dz3
    dz2 = da2 * (z2 > 0)
    dw2 = (dz2 @ a1.T) / m
    db2 = np.sum(dz2, axis=1, keepdims=True)

    # 1st layer (tanh)

    da1 = w2.T @ dz2
    dz1 = da1 * (1 - a1**2)
    dw1 = (dz1 @ X.T) / m
    db1 = np.sum(dz1, axis=1, keepdims=True)

    grades = {"dw1":dw1,"db1": db1, "dw2":dw2, "db2": db2, "dw3":dw3, "db3": db3}
    
    return grades

In [62]:
def update(params, grades, lr=0.01):
    W1 = params['w1']
    w2 = params['w2']
    w3 = params['w3']

    b1 = params['b1']
    b2 = params['b2']
    b3 = params['b3']


    dw1 = grades["dw1"]
    db1 = grades["db1"]
    dw2 = grades["dw2"]
    db2 = grades["db2"]
    dw3 = grades["dw3"]
    db3 = grades["db3"]
 
 
    W1 = W1 - lr*dw1
    b1 = b1 - lr*db1
    w2 = w2 - lr*dw2
    b2 = b2 - lr*db2
    w3 = w3 - lr*dw3
    b3 = b3 - lr*db3

    params = {'w1':W1,   
              "b1": b1,
            "w2": w2,
            "b2": b2,
            "w3": w3,
            "b3": b3}
    
    return params

In [80]:
def NN(X,y,hidden_nodes,itr=10000,print_cost=False):
    np.random.seed(3)
    nx, nh1, nh2, ny = layer_sizes(X,y,hidden_nodes)
    params = int_params(nx, nh1, nh2, ny)
    for i in range(0, itr):
        a3,cache = fwp(X,params)
        cost = compute_cost(y, a3)
        grades = bwp(X, y, params, cache)
        params = update(params, grades, lr= 0.01)

        if print_cost and i % 10000 == 0:
            print("cost %i: %f" % (i, cost))
    return params

In [81]:
NN(X,y,hidden_nodes,itr=100000,print_cost=True)

cost 0: 0.750617
cost 10000: 0.004668
cost 20000: 0.002086
cost 30000: 0.001316
cost 40000: 0.000952
cost 50000: 0.000742
cost 60000: 0.000606
cost 70000: 0.000511
cost 80000: 0.000441
cost 90000: 0.000387


{'w1': array([[ 1.48856176,  0.67169686,  0.32985578, -1.99025058],
        [-0.23927342, -0.02116411, -0.05797549, -0.73523498],
        [ 0.00540474, -0.40156964, -1.59201298,  0.91232345],
        [ 0.5510446 ,  2.08510413, -0.22170108, -0.44027511]]),
 'b1': array([[ 2.81158735],
        [-1.90191211],
        [-1.39462353],
        [-0.44241026]]),
 'w2': array([[-0.34283612, -2.20332586,  0.0955168 , -2.65033047],
        [-1.69167674, -0.46862967,  2.10371986,  0.09491971],
        [-1.66316648, -0.90509196,  2.08416821, -0.02815967]]),
 'b2': array([[ 1.14461747],
        [ 0.3400996 ],
        [-0.21451063]]),
 'w3': array([[-2.61157813,  1.92651002,  2.51185441]]),
 'b3': array([[6.69620846]])}